# 12. 침묵 그룹 메타데이터 기반 실패 위험 요인 분석

**분석 목적:** 리뷰 0~9개인 `침묵` 그룹과 리뷰 10~49개인 `유저 반응` 그룹을 전체 데이터 기준으로 비교해, 유저의 첫 반응 확보에 실패할 가능성이 높은 메타데이터 조건을 탐색한다.

**핵심 질문:** 리뷰 텍스트 없이도 장르·가격·플랫폼·카테고리·상점 설명 같은 메타데이터에서 첫 반응 실패 위험 요인을 찾을 수 있는가?

**사용 데이터**
- `data/preprocessed/steam_indie_games_silence.csv`: 침묵 그룹, 리뷰 0~9개
- `data/preprocessed/steam_indie_games.csv`: 유저 반응 그룹, 리뷰 10~49개

**해석 원칙:** 이 분석은 실패 원인을 확정하는 인과 분석이 아니라, 침묵 그룹에서 과대표현되는 속성을 통해 출시 전 점검해야 할 위험 신호를 찾는 탐색 분석이다.

## 분석 흐름

1. 데이터 로드 및 그룹 정의
2. 침묵 정도 분포 확인
3. 장르별 침묵 과대표현 분석
4. 가격대별 침묵 비율 분석
5. 플랫폼·카테고리 속성 비교
6. 상점 설명 길이 비교
7. 실패 위험 요인 후보 요약

In [1]:
import ast
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## 1. 데이터 로드 및 그룹 정의

두 데이터셋은 `00_preprocessing.ipynb`에서 동일한 기간·EA/F2P/price=0·부적합 장르 필터를 적용한 결과다.

In [2]:
DATA_DIR = Path("../../../data/preprocessed")
SILENCE_PATH = DATA_DIR / "steam_indie_games_silence.csv"
RESPONSE_PATH = DATA_DIR / "steam_indie_games_graded.csv"

if not SILENCE_PATH.exists():
    raise FileNotFoundError(f"침묵 그룹 데이터가 없습니다: {SILENCE_PATH}")
if not RESPONSE_PATH.exists():
    raise FileNotFoundError(f"유저 반응 그룹 데이터가 없습니다: {RESPONSE_PATH}")

df_silence = pd.read_csv(SILENCE_PATH)
df_response = pd.read_csv(RESPONSE_PATH)
df_response = df_response[df_response["total_reviews"].between(10, 49)].copy()

def parse_genres(value: str) -> list[str]:
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []

def parse_tags(value):
    if pd.isna(value):
        return []
    if isinstance(value, dict):
        return list(value.keys())
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            if isinstance(parsed, dict):
                return list(parsed.keys())
            if isinstance(parsed, list):
                return parsed
        except Exception:
            try:
                parsed = ast.literal_eval(value)
                if isinstance(parsed, dict):
                    return list(parsed.keys())
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                return []
    return []

for frame in [df_silence, df_response]:
    frame["genres"] = frame["genres"].apply(parse_genres)
    frame["tag_list"] = frame["tags"].apply(parse_tags) if "tags" in frame.columns else [[] for _ in range(len(frame))]

df_silence["response_group"] = "침묵 (리뷰 0~9개)"
df_response["response_group"] = "유저 반응 (리뷰 10~49개)"

df = pd.concat([df_silence, df_response], ignore_index=True)

for col in ["price", "total_reviews", "owners_lower", "owners_higher", "recommendations_total", "achievements_total"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["is_silence"] = df["response_group"].eq("침묵 (리뷰 0~9개)")

overall_silence_rate = df['is_silence'].mean() * 100

GROUP_ORDER = ["침묵 (리뷰 0~9개)", "유저 반응 (리뷰 10~49개)"]
GROUP_COLOR = {
    "침묵 (리뷰 0~9개)": "#C44E52",
    "유저 반응 (리뷰 10~49개)": "#4C72B0",
}

print(f"침묵 그룹: {len(df_silence):,}개")
print(f"유저 반응 그룹: {len(df_response):,}개")
print(f"전체: {len(df):,}개")
print(f"태그 보유율(침묵): {df_silence['tag_list'].apply(bool).mean()*100:.1f}%")
print(f"태그 보유율(유저 반응): {df_response['tag_list'].apply(bool).mean()*100:.1f}%")

침묵 그룹: 6,737개
유저 반응 그룹: 4,904개
전체: 11,641개
release_date 결측: 0개
price=0: 0개


,appid,positive,negative,price,genres,total_reviews,name,developers,release_date,short_description,publishers,categories,windows,mac,linux,recommendations_total,achievements_total,owners_lower,owners_higher,tags,response_group,release_year,is_silence
0,370630,1,0,6.99,"['Adventure', 'Casual', 'Indie', 'RPG', 'Strat...",1,Sylia,Aldorlea Games,2025-04-30,Pets vs Aliens! When your heroes are not good ...,Aldorlea Games,"Single-player, Steam Trading Cards, Steam Clou...",True,False,False,NaN,NaN,20000,50000,"{""2D"": 361, ""Dog"": 461, ""RPG"": 450, ""CRPG"": 38...",침묵 (리뷰 0~9개),2025,True
1,400390,6,0,11.99,"['Casual', 'Indie']",6,Delusion,Silver Line Games,2023-05-26,Delusion is an unusual puzzle game that tells ...,Silver Line Games,"Single-player, Steam Achievements, Full contro...",True,True,True,NaN,16.0,0,20000,"{""2.5D"": 84, ""Indie"": 45, ""Logic"": 57, ""Space""...",침묵 (리뷰 0~9개),2023,True
2,520410,1,1,2.99,"['Action', 'Indie']",2,SpacePod,Robusta Gameworks,2023-05-04,Classic SpacePod now available on Steam!,Fabulous,"Single-player, Family Sharing",True,False,False,NaN,NaN,0,20000,"{""3D"": 33, ""Indie"": 20, ""Action"": 65, ""Combat""...",침묵 (리뷰 0~9개),2023,True
3,544690,2,1,0.99,"['Adventure', 'Casual', 'Indie']",3,Kriaturaz - O Guardião das Lendas (Base),Messier Games & Animations,2023-03-18,This is a Open Source Game. Kriaturaz is a gam...,Messier Games & Animations,"Single-player, Family Sharing",True,False,False,NaN,NaN,0,20000,"{""3D"": 24, ""PvE"": 61, ""Cute"": 109, ""Indie"": 19...",침묵 (리뷰 0~9개),2023,True
4,547670,8,0,9.99,"['Casual', 'Indie', 'Simulation']",8,Musical Range,Rockhopper Studios,2024-04-02,Are you ready to play music for a crowd going ...,Rockhopper Studios,"Single-player, Tracked Controller Support, VR ...",True,False,False,NaN,NaN,0,20000,"{""VR"": 14, ""Indie"": 32, ""Music"": 13, ""Casual"":...",침묵 (리뷰 0~9개),2024,True


In [3]:
group_summary = (
    df.groupby("response_group")
    .agg(
        game_count=("appid", "nunique"),
        median_reviews=("total_reviews", "median"),
        avg_reviews=("total_reviews", "mean"),
        median_price=("price", "median"),
        median_owners_lower=("owners_lower", "median"),
        median_owners_higher=("owners_higher", "median"),
    )
    .reindex(GROUP_ORDER)
    .reset_index()
)
group_summary["ratio"] = group_summary["game_count"] / group_summary["game_count"].sum() * 100
group_summary.round(2)

,response_group,game_count,median_reviews,avg_reviews,median_price,median_owners_lower,median_owners_higher,ratio
0,침묵 (리뷰 0~9개),6737,3.0,3.79,3.99,0.0,20000.0,57.87
1,유저 반응 (리뷰 10~49개),4904,20.0,22.58,4.99,0.0,20000.0,42.13


In [4]:
fig = px.bar(
    group_summary,
    x="response_group",
    y="game_count",
    color="response_group",
    color_discrete_map=GROUP_COLOR,
    text="ratio",
    title="전체 분석 대상 중 침묵/유저 반응 그룹 비중",
    labels={"response_group": "그룹", "game_count": "게임 수"},
    category_orders={"response_group": GROUP_ORDER},
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(template="plotly_white", showlegend=False, width=850, height=480)
fig.show()

## 2. 침묵 정도 분포

리뷰 0~9개는 모두 같은 침묵 상태가 아니다. 0개는 완전 침묵이고, 7~9개는 리뷰 10개 기준에 가까운 경계 반응이다.

In [5]:
def assign_silence_level(total_reviews: int) -> str:
    if total_reviews == 0:
        return "0개: 완전 침묵"
    if total_reviews <= 3:
        return "1~3개: 극소 반응"
    if total_reviews <= 6:
        return "4~6개: 약한 반응"
    if total_reviews <= 9:
        return "7~9개: 경계 반응"
    return "10~49개: 유저 반응"

SILENCE_LEVEL_ORDER = [
    "0개: 완전 침묵",
    "1~3개: 극소 반응",
    "4~6개: 약한 반응",
    "7~9개: 경계 반응",
    "10~49개: 유저 반응",
]

df["silence_level"] = df["total_reviews"].apply(assign_silence_level)
df["silence_level"] = pd.Categorical(df["silence_level"], categories=SILENCE_LEVEL_ORDER, ordered=True)

silence_level_counts = (
    df.groupby("silence_level", observed=True)
    .agg(game_count=("appid", "nunique"))
    .reset_index()
)
silence_level_counts["ratio"] = silence_level_counts["game_count"] / silence_level_counts["game_count"].sum() * 100
silence_level_counts

,silence_level,game_count,ratio
0,0개: 완전 침묵,143,1.228417
1,1~3개: 극소 반응,3426,29.430461
2,4~6개: 약한 반응,1995,17.137703
3,7~9개: 경계 반응,1173,10.076454
4,10~49개: 유저 반응,4904,42.126965


In [6]:
fig = px.bar(
    silence_level_counts,
    x="silence_level",
    y="game_count",
    text="ratio",
    title="리뷰 수 기준 침묵 정도 분포",
    labels={"silence_level": "침묵 정도", "game_count": "게임 수"},
    color="silence_level",
    color_discrete_sequence=px.colors.sequential.Reds_r,
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside", showlegend=False)
fig.update_layout(template="plotly_white", width=1000, height=500)
fig.show()

## 3. 장르별 침묵 과대표현 분석

장르는 다중 장르를 모두 반영한다. 각 장르에 대해 “그 장르를 포함한 게임 중 침묵 그룹 비율”을 계산한다.

In [7]:
def parse_genres(value) -> list[str]:
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return value
    if isinstance(value, str):
        return ast.literal_eval(value)
    return []

TARGET_GENRES = ["Action", "Adventure", "Casual", "RPG", "Simulation", "Strategy", "Sports", "Racing"]

genre_df = df[["appid", "response_group", "genres"]].copy()
genre_df["genre_list"] = genre_df["genres"].apply(parse_genres)
genre_df = genre_df.explode("genre_list").rename(columns={"genre_list": "genre"})
genre_df = genre_df[genre_df["genre"].isin(TARGET_GENRES)].copy()

genre_group = (
    genre_df.groupby(["genre", "response_group"])["appid"]
    .count()
    .unstack(fill_value=0)
    .reindex(columns=GROUP_ORDER, fill_value=0)
)
genre_group["total_games"] = genre_group.sum(axis=1)
genre_group["silence_games"] = genre_group["침묵 (리뷰 0~9개)"]
genre_group["response_games"] = genre_group["유저 반응 (리뷰 10~49개)"]
genre_group["silence_rate"] = genre_group["silence_games"] / genre_group["total_games"] * 100
genre_group["response_rate"] = genre_group["response_games"] / genre_group["total_games"] * 100
genre_group["lift"] = genre_group["silence_rate"] / (df["is_silence"].mean() * 100)
genre_risk = genre_group.reset_index().sort_values("silence_rate", ascending=False)
display(genre_risk.round(2))

,genre,game_count,silence_games,median_price,median_owners_lower,silence_rate,rate_diff,lift
2,Casual,5955,3625,3.99,0.0,60.87,3.00,1.05
4,Racing,454,276,4.99,0.0,60.79,2.92,1.05
7,Strategy,2366,1421,4.99,0.0,60.06,2.19,1.04
0,Action,5282,3068,4.99,0.0,58.08,0.21,1.00
6,Sports,438,249,4.99,0.0,56.85,-1.02,0.98
3,RPG,2057,1153,4.99,0.0,56.05,-1.82,0.97
1,Adventure,5356,2884,4.99,0.0,53.85,-4.03,0.93
5,Simulation,2359,1244,4.99,0.0,52.73,-5.14,0.91


In [8]:
plot_genre = genre_risk.sort_values('silence_rate', ascending=True)

fig = px.bar(
    plot_genre,
    x='silence_rate',
    y='genre',
    orientation='h',
    color='lift',
    color_continuous_scale='Reds',
    title='장르별 침묵 비율과 침묵 Lift',
    labels={'genre': '장르', 'silence_rate': '침묵 비율(%)', 'lift': '침묵 Lift'},
    hover_data={'total_games': True, 'silence_games': True, 'response_games': True, 'response_rate': ':.1f'}
)
fig.update_layout(template='plotly_white', width=900, height=500, coloraxis_showscale=False)
fig.show()

## 5. 가격대별 침묵 비율

`price=0`은 전처리에서 제외되었으므로, 여기서는 유료 게임 가격대 안에서 첫 반응 실패 위험이 달라지는지 확인한다. 극단적 가격은 별도로 진단하고, 일반적인 가격대 비교는 $60 이하로 제한한다.

## 4. 태그별 침묵 vs 유저 반응 Lift 비교

침묵 그룹과 유저 반응 그룹 모두에서 태그가 확보되었으므로, 이제 첫 반응 확보와 연결되는 태그를 직접 비교할 수 있다. 여기서는 각 그룹 내부 태그 비율을 전체 태그 비율과 비교한 Lift를 사용한다.

- `Lift > 1`: 해당 그룹에서 기대보다 더 자주 나타나는 태그
- `침묵 Lift`가 높음: 첫 반응 확보 실패와 함께 나타날 가능성이 큰 태그
- `유저 반응 Lift`가 높음: 리뷰 10~49개 반응 확보와 함께 나타날 가능성이 큰 태그

In [ ]:
MIN_TAG_GAMES = 50
GENERAL_TAGS = set(TARGET_GENRES + ['Indie', 'Singleplayer', 'Multiplayer', '2D', '3D'])

tag_df = df[['appid', 'response_group', 'tag_list']].explode('tag_list').dropna(subset=['tag_list']).copy()
tag_df = tag_df[tag_df['tag_list'].astype(str).str.len() > 0]

overall_tag_rate = (
    tag_df.groupby('tag_list')['appid'].count() / tag_df['appid'].nunique()
).rename('overall_tag_rate')

group_tag = (
    tag_df.groupby(['response_group', 'tag_list'])['appid'].count().rename('tag_games').reset_index()
)
group_totals = df.groupby('response_group')['appid'].nunique().rename('group_games').reset_index()

tag_lift = group_tag.merge(group_totals, on='response_group', how='left')
tag_lift = tag_lift.merge(overall_tag_rate.reset_index(), on='tag_list', how='left')
tag_lift['group_tag_rate'] = tag_lift['tag_games'] / tag_lift['group_games']
tag_lift['lift'] = tag_lift['group_tag_rate'] / tag_lift['overall_tag_rate']
tag_lift = tag_lift[(tag_lift['tag_games'] >= MIN_TAG_GAMES) & (~tag_lift['tag_list'].isin(GENERAL_TAGS))].copy()

top_silence_tags = tag_lift[tag_lift['response_group'] == '침묵 (리뷰 0~9개)'].sort_values(['lift', 'tag_games'], ascending=[False, False]).head(15)
top_response_tags = tag_lift[tag_lift['response_group'] == '유저 반응 (리뷰 10~49개)'].sort_values(['lift', 'tag_games'], ascending=[False, False]).head(15)

display(top_silence_tags[['tag_list', 'tag_games', 'group_tag_rate', 'lift']].round(3))
display(top_response_tags[['tag_list', 'tag_games', 'group_tag_rate', 'lift']].round(3))

plot_tags = pd.concat([top_silence_tags, top_response_tags], ignore_index=True)
fig = px.bar(
    plot_tags,
    x='lift',
    y='tag_list',
    color='response_group',
    orientation='h',
    barmode='group',
    title='침묵 vs 유저 반응 그룹의 시그니처 태그 Lift',
    labels={'tag_list': '태그', 'lift': 'Lift', 'response_group': '그룹'},
    color_discrete_map=GROUP_COLOR,
    category_orders={'response_group': GROUP_ORDER},
    hover_data={'tag_games': True, 'group_tag_rate': ':.3f'}
)
fig.update_layout(template='plotly_white', width=1000, height=700)
fig.show()

**해석:** 태그 Lift는 '무슨 태그가 많은가'보다 '어떤 태그가 특정 그룹에서 유난히 더 자주 보이는가'를 보여준다. 침묵 Lift가 높은 태그는 첫 반응 확보에 불리했을 가능성이 있고, 유저 반응 Lift가 높은 태그는 최소한 리뷰 10~49개 수준의 반응 확보와 연결된 태그로 해석할 수 있다.

In [9]:
price_diag = (
    df.assign(price_over_60=df["price"] > 60)
    .groupby("response_group")
    .agg(
        game_count=("appid", "nunique"),
        over_60_games=("price_over_60", "sum"),
        max_price=("price", "max"),
        median_price=("price", "median"),
    )
    .reindex(GROUP_ORDER)
    .reset_index()
)
price_diag["over_60_rate"] = price_diag["over_60_games"] / price_diag["game_count"] * 100
price_diag.round(2)

,response_group,game_count,over_60_games,max_price,median_price,over_60_rate
0,침묵 (리뷰 0~9개),6737,16,999.98,3.99,0.24
1,유저 반응 (리뷰 10~49개),4904,19,199.99,4.99,0.39


In [10]:
price_df = df[(df["price"] > 0) & (df["price"] <= 60)].copy()
PRICE_BINS = [0, 5, 10, 15, 20, 30, 60]
PRICE_LABELS = ["~$5", "$5~10", "$10~15", "$15~20", "$20~30", "$30~60"]
price_df["price_range"] = pd.cut(price_df["price"], bins=PRICE_BINS, labels=PRICE_LABELS, right=True)

price_risk = (
    price_df.groupby("price_range", observed=True)
    .agg(
        game_count=("appid", "nunique"),
        silence_games=("is_silence", "sum"),
        median_reviews=("total_reviews", "median"),
    )
    .reset_index()
)
price_risk["silence_rate"] = price_risk["silence_games"] / price_risk["game_count"] * 100
price_risk["rate_diff"] = price_risk["silence_rate"] - overall_silence_rate
price_risk["lift"] = price_risk["silence_rate"] / overall_silence_rate

price_risk.round(2)

,price_range,game_count,silence_games,median_reviews,silence_rate,rate_diff,lift
0,~$5,7488,4732,6.0,63.19,5.32,1.09
1,$5~10,2801,1424,9.0,50.84,-7.03,0.88
2,$10~15,831,351,12.0,42.24,-15.63,0.73
3,$15~20,323,133,13.0,41.18,-16.70,0.71
4,$20~30,121,54,13.0,44.63,-13.24,0.77
5,$30~60,42,27,2.0,64.29,6.41,1.11


In [11]:
fig = px.bar(
    price_risk,
    x="price_range",
    y="silence_rate",
    color="rate_diff",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    text="silence_rate",
    title="가격대별 침묵 비율 ($60 이하 유료 게임)",
    labels={"price_range": "가격대", "silence_rate": "침묵 비율(%)", "rate_diff": "전체 대비 차이(%p)"},
    hover_data={"game_count": ":,", "silence_games": ":,", "lift": ":.2f"},
)
fig.add_hline(y=overall_silence_rate, line_dash="dash", line_color="#333333", annotation_text="전체 침묵 비율")
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(template="plotly_white", width=950, height=520, yaxis=dict(ticksuffix="%"))
fig.show()

In [12]:
fig = px.box(
    price_df,
    x="response_group",
    y="price",
    color="response_group",
    color_discrete_map=GROUP_COLOR,
    category_orders={"response_group": GROUP_ORDER},
    points=False,
    title="침묵 vs 유저 반응 그룹 가격 분포 ($60 이하)",
    labels={"response_group": "그룹", "price": "가격 (USD)"},
)
fig.update_layout(template="plotly_white", showlegend=False, width=850, height=520)
fig.update_yaxes(tickprefix="$")
fig.show()

## 6. 플랫폼 지원과 카테고리 속성

플랫폼 지원, 컨트롤러 지원, Steam Achievements, Steam Cloud 같은 상점 속성은 초기 구매·플레이 진입 장벽에 영향을 줄 수 있다.

In [13]:
def to_bool(value) -> bool:
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() == "true"

for col in ["windows", "mac", "linux"]:
    df[col] = df[col].apply(to_bool)

df["platform_count"] = df[["windows", "mac", "linux"]].sum(axis=1)
df["supports_mac_or_linux"] = df["mac"] | df["linux"]

platform_features = [
    ("windows", "Windows 지원"),
    ("mac", "Mac 지원"),
    ("linux", "Linux 지원"),
    ("supports_mac_or_linux", "Mac/Linux 중 하나 이상"),
]

platform_rows = []
for col, label in platform_features:
    total = df[col].sum()
    silence = df.loc[df[col], "is_silence"].sum()
    rate = silence / total * 100 if total else 0
    platform_rows.append({"feature": label, "game_count": int(total), "silence_games": int(silence), "silence_rate": rate})

platform_risk = pd.DataFrame(platform_rows)
platform_risk["rate_diff"] = platform_risk["silence_rate"] - overall_silence_rate
platform_risk["lift"] = platform_risk["silence_rate"] / overall_silence_rate
platform_risk.round(2)

,feature,game_count,silence_games,silence_rate,rate_diff,lift
0,Windows 지원,11564,6681,57.77,-0.10,1.00
1,Mac 지원,1575,787,49.97,-7.90,0.86
2,Linux 지원,1392,725,52.08,-5.79,0.90
3,Mac/Linux 중 하나 이상,2196,1148,52.28,-5.60,0.90


In [14]:
fig = px.bar(
    platform_risk.sort_values("silence_rate"),
    x="silence_rate",
    y="feature",
    orientation="h",
    color="rate_diff",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    text="silence_rate",
    title="플랫폼 지원 속성별 침묵 비율",
    labels={"silence_rate": "침묵 비율(%)", "feature": "속성", "rate_diff": "전체 대비 차이(%p)"},
    hover_data={"game_count": ":,", "silence_games": ":,", "lift": ":.2f"},
)
fig.add_vline(x=overall_silence_rate, line_dash="dash", line_color="#333333", annotation_text="전체 침묵 비율")
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(template="plotly_white", width=950, height=420)
fig.show()

In [15]:
CATEGORY_KEYWORDS = {
    "Single-player": "Single-player",
    "Multi-player": "Multi-player",
    "Co-op": "Co-op",
    "Online Co-op": "Online Co-op",
    "Steam Achievements": "Steam Achievements",
    "Steam Cloud": "Steam Cloud",
    "Full controller support": "Full controller support",
    "Partial Controller Support": "Partial Controller Support",
    "Family Sharing": "Family Sharing",
    "Steam Leaderboards": "Steam Leaderboards",
}

category_rows = []
category_text = df["categories"].fillna("").astype(str)
for label, keyword in CATEGORY_KEYWORDS.items():
    mask = category_text.str.contains(keyword, case=False, regex=False, na=False)
    total = mask.sum()
    silence = df.loc[mask, "is_silence"].sum()
    rate = silence / total * 100 if total else 0
    category_rows.append({"feature": label, "game_count": int(total), "silence_games": int(silence), "silence_rate": rate})

category_risk = pd.DataFrame(category_rows)
category_risk["rate_diff"] = category_risk["silence_rate"] - overall_silence_rate
category_risk["lift"] = category_risk["silence_rate"] / overall_silence_rate
category_risk.sort_values("silence_rate", ascending=False).round(2)

,feature,game_count,silence_games,silence_rate,rate_diff,lift
8,Family Sharing,11561,6681,57.79,-0.08,1.00
0,Single-player,11363,6560,57.73,-0.14,1.00
7,Partial Controller Support,1278,736,57.59,-0.28,1.00
1,Multi-player,1308,741,56.65,-1.22,0.98
2,Co-op,805,438,54.41,-3.46,0.94
9,Steam Leaderboards,907,467,51.49,-6.38,0.89
4,Steam Achievements,6114,3094,50.61,-7.27,0.87
3,Online Co-op,420,211,50.24,-7.63,0.87
6,Full controller support,3084,1494,48.44,-9.43,0.84
5,Steam Cloud,2937,1348,45.90,-11.98,0.79


In [16]:
fig = px.bar(
    category_risk.sort_values("silence_rate", ascending=True),
    x="silence_rate",
    y="feature",
    orientation="h",
    color="rate_diff",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    text="silence_rate",
    title="Steam 카테고리 속성별 침묵 비율",
    labels={"silence_rate": "침묵 비율(%)", "feature": "카테고리 속성", "rate_diff": "전체 대비 차이(%p)"},
    hover_data={"game_count": ":,", "silence_games": ":,", "lift": ":.2f"},
)
fig.add_vline(x=overall_silence_rate, line_dash="dash", line_color="#333333", annotation_text="전체 침묵 비율")
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(template="plotly_white", width=950, height=620)
fig.show()

## 7. 상점 설명 길이 비교

짧거나 정보량이 부족한 설명은 게임의 매력 전달에 불리할 수 있다. 여기서는 `short_description` 길이를 간단한 대리 지표로 사용한다.

In [17]:
df["description_length"] = df["short_description"].fillna("").astype(str).str.len()

DESC_BINS = [-1, 0, 80, 160, 240, 500, float("inf")]
DESC_LABELS = ["설명 없음", "1~80자", "81~160자", "161~240자", "241~500자", "500자 초과"]
df["description_length_bin"] = pd.cut(df["description_length"], bins=DESC_BINS, labels=DESC_LABELS)

desc_risk = (
    df.groupby("description_length_bin", observed=True)
    .agg(
        game_count=("appid", "nunique"),
        silence_games=("is_silence", "sum"),
        median_reviews=("total_reviews", "median"),
    )
    .reset_index()
)
desc_risk["silence_rate"] = desc_risk["silence_games"] / desc_risk["game_count"] * 100
desc_risk["rate_diff"] = desc_risk["silence_rate"] - overall_silence_rate
desc_risk["lift"] = desc_risk["silence_rate"] / overall_silence_rate

desc_risk.round(2)

,description_length_bin,game_count,silence_games,median_reviews,silence_rate,rate_diff,lift
0,설명 없음,80,59,4.5,73.75,15.88,1.27
1,1~80자,836,547,5.0,65.43,7.56,1.13
2,81~160자,2577,1520,7.0,58.98,1.11,1.02
3,161~240자,3865,2212,7.0,57.23,-0.64,0.99
4,241~500자,4283,2399,8.0,56.01,-1.86,0.97


In [18]:
fig = px.bar(
    desc_risk,
    x="description_length_bin",
    y="silence_rate",
    color="rate_diff",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    text="silence_rate",
    title="상점 설명 길이 구간별 침묵 비율",
    labels={"description_length_bin": "설명 길이", "silence_rate": "침묵 비율(%)", "rate_diff": "전체 대비 차이(%p)"},
    hover_data={"game_count": ":,", "silence_games": ":,", "lift": ":.2f"},
)
fig.add_hline(y=overall_silence_rate, line_dash="dash", line_color="#333333", annotation_text="전체 침묵 비율")
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(template="plotly_white", width=950, height=520, yaxis=dict(ticksuffix="%"))
fig.show()

In [19]:
fig = px.box(
    df,
    x="response_group",
    y="description_length",
    color="response_group",
    color_discrete_map=GROUP_COLOR,
    category_orders={"response_group": GROUP_ORDER},
    points=False,
    title="침묵 vs 유저 반응 그룹의 상점 설명 길이 분포",
    labels={"response_group": "그룹", "description_length": "short_description 글자 수"},
)
fig.update_layout(template="plotly_white", showlegend=False, width=850, height=520)
fig.show()

## 8. 실패 위험 요인 후보 요약

각 분석 축에서 침묵 비율이 전체 평균보다 높은 속성을 모아, 출시 전 점검해야 할 위험 신호 후보로 정리한다.

In [20]:
def top_risk_rows(frame: pd.DataFrame, label_col: str, source: str, top_n: int = 5) -> pd.DataFrame:
    count_col = 'total_games' if 'total_games' in frame.columns else 'game_count'
    out = frame.nlargest(top_n, 'silence_rate')[[label_col, 'silence_rate', 'lift', count_col]].copy()
    out.columns = ['attribute', 'silence_rate', 'lift', 'game_count']
    out['source'] = source
    return out

risk_summary = pd.concat([
    top_risk_rows(genre_risk, 'genre', '장르'),
    top_risk_rows(price_risk, 'price_range', '가격대'),
    top_risk_rows(platform_risk, 'feature', '플랫폼'),
    top_risk_rows(category_risk, 'feature', '카테고리'),
    top_risk_rows(desc_risk, 'description_length_bin', '설명 길이'),
], ignore_index=True)
display(risk_summary)

fig = px.scatter(
    risk_summary,
    x='lift',
    y='silence_rate',
    size='game_count',
    color='source',
    text='attribute',
    title='침묵 위험 신호 요약: 침묵 비율 × Lift',
    labels={'lift': '침묵 Lift', 'silence_rate': '침묵 비율(%)', 'source': '속성 유형'},
)
fig.update_traces(textposition='top center')
fig.update_layout(template='plotly_white', width=1100, height=700)
fig.show()

,analysis_axis,attribute,game_count,silence_games,silence_rate,rate_diff,lift
0,장르,Casual,5955,3625,60.87,3.00,1.05
1,장르,Racing,454,276,60.79,2.92,1.05
2,장르,Strategy,2366,1421,60.06,2.19,1.04
3,장르,Action,5282,3068,58.08,0.21,1.00
4,장르,Sports,438,249,56.85,-1.02,0.98
5,가격대,$30~60,42,27,64.29,6.41,1.11
6,가격대,~$5,7488,4732,63.19,5.32,1.09
7,가격대,$5~10,2801,1424,50.84,-7.03,0.88
8,가격대,$20~30,121,54,44.63,-13.24,0.77
9,가격대,$10~15,831,351,42.24,-15.63,0.73


In [21]:
fig = px.scatter(
    risk_summary,
    x='lift',
    y='silence_rate',
    size='game_count',
    color='source',
    text='attribute',
    title='침묵 과대표현 위험 신호 후보',
    labels={'lift': '침묵 Lift', 'silence_rate': '침묵 비율(%)', 'game_count': '게임 수', 'source': '분석 축'},
)
fig.add_hline(y=overall_silence_rate, line_dash='dash', line_color='#666666')
fig.add_vline(x=1, line_dash='dash', line_color='#666666')
fig.update_traces(textposition='top center', marker=dict(opacity=0.75))
fig.update_layout(template='plotly_white', width=1050, height=650)
fig.show()

## 종합 해석

- 침묵 비율이 높은 속성은 “그 속성이 실패 원인”이라는 뜻이 아니라, 첫 반응 확보에 실패한 게임에서 상대적으로 자주 관찰되는 조건이다.
- 장르와 가격은 함께 해석해야 한다. 같은 가격이라도 장르별 기대 콘텐츠 규모와 경쟁 강도가 다르다.
- 플랫폼·카테고리·설명 길이는 상점 페이지에서 유저가 구매 전 확인하는 정보이므로, 리뷰 10~49개 수준의 첫 유저 반응을 얻기 위한 최소 준비 상태를 점검하는 데 유용하다.
- 태그 데이터는 침묵 그룹에서 아직 미수집 상태이므로, 태그 기반 실패 위험 분석은 추가 수집 이후 수행하는 것이 안전하다.

**실무적 결론:** 출시 전에는 “어떤 장르/가격대인가”뿐 아니라, 해당 장르에서 유저가 기대하는 플랫폼 지원, 카테고리 기능, 상점 설명의 정보량을 함께 점검해야 한다. 침묵 그룹에서 과대표현되는 속성은 출시 전 체크리스트의 우선 검토 항목으로 활용한다.